# Projet analyse de données

## Contexte

**Auteurs :** Boely Jocelin, Lila Costadau, Felix Decker, Lukas Fauli

**Date :** 18/05/2026

**Objectif de ce notebook :** Ce notebook présente le traitement initial des données, l'étude d'ouliers et cherche à répondre à la problématique suivantes:
- Est-ce qu'on peut regrouper les jeux dans des classes homogènes?

## Librairies utilisées

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Fonctions longues de plot dans le sous dossier Fonctions
from Fonctions.plot_corr_circles import plot_correlation_circle
from Fonctions.plot_outliers import plot_outliers
from Fonctions.plot_outliers import plot_outliers_non_nuls
from Fonctions.partition_analysis import partition_analysis
from Fonctions.partition_analysis import collect_cluster_results
from Fonctions.plot_cluster_analysis import plot_one_Variable_on_umap
from Fonctions.plot_cluster_analysis import make_labels

## Récupération des données

In [ ]:
df = pd.read_csv('./Downloads/games.csv')



df.columns = ['Name', 'Release date', 'Estimated owners', 'Peak CCU',
       'Required age', 'Price', 'Discount', 'DLC count', 'About the game',
       'Supported languages', 'Full audio languages', 'Reviews',
       'Header image', 'Website', 'Support url', 'Support email', 'Windows',
       'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score',
       'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations',
       'Notes', 'Average playtime forever', 'Average playtime two weeks',
       'Median playtime forever', 'Median playtime two weeks', 'Developers',
       'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']

In [ ]:
inutiles = ["About the game", "Reviews", "Website", "Support url", "Support email", 
            "Metacritic url", "Notes", "Score rank", "Movies", "Header image", "Screenshots", "User score"]
df = df.drop(columns=inutiles)
df['Required age'] = df['Required age'].astype('object') 
df.info()

Nous décidons de considérer **Required age** comme variable qualitative et supprimons toutes les colonnes "inutiles" ou contenant trop peu de valeurs, particulièrement **Score rank** et **User Score**.

Le dataset sur Kaggle présente une erreur dans l'entête du CSV, il manque une virgule entre le titre de deux colonnes (Discount et DLC count). Pandas et Python peuvent facilement rectifier cette erreur en renommant les colonnes. En R, il a été choisi de simplement ajouter la virgule manquante dans le document CSV

## Conversion des langages en nombre de langages.

La première transformation que nous réalisons est le regroupement et la transformation des variables qualitatives **Supported languages** et **Full audio languages** en une seule variable quantitative qui compte le nombre de langages (uniques) disponibles dans chaque jeu. 

Cela nous permet de garder une partie de l'information disponible par ces variables et de réduire fortement le nombre de modalités qualitatives.

In [ ]:
import re

def count_languages(row):
    langs = set()
    for col in ["Supported languages", "Full audio languages"]:
        val = str(row[col])
        # Extraire le contenu entre [ et ]
        matches = re.findall(r'\[([^\]]*)\]', val)
        for match in matches:
            if match.strip():
                # Compter les éléments par les virgules
                items = [x.strip().strip("'") for x in match.split(',')]
                langs.update(items)
    return len(langs)

df["Language Count"] = df.apply(count_languages, axis=1)

## Suppression des doublons

Nous supprimons les lignes où les noms, les développeurs et les éditeurs sont les mêmes, car nous les considérons comme étant le même jeu.

Nous conservons uniquement celle avec le pic de CCU le plus élevé comme 'représentant' de ce jeu.

In [ ]:
before = df.shape
df = df.loc[
    df.groupby(['Name', 'Developers', 'Publishers'], dropna=False)['Peak CCU'].idxmax()
]
print(before[0] - df.shape[0])

Nous avons donc supprimé ~150 jeux qui était très probablement des duplicatas.

## Suppression des jeux, sans Tags, Genres, Categories et Language

Nous considérons que les jeux qui ne possèdent pas de Tags, Genres, Categories et avec Language Count à 0 sont probablement des jeux qui ne sont pas encore publiés ou bien des erreurs, nous décidons donc de les supprimer du dataset.

In [ ]:
before = df.shape
df = df[
    (df["Tags"] != "") &
    (df["Genres"] != "") &
    (df["Categories"] != "") &
    (df["Language Count"] != 0)
]
print(before[0] - df.shape[0])

Nous avons donc supprimé au total 8500+ jeux sur un total de 122000 jeux initialement, soit environ 7% du dataset. Nous considérons que nous n'avons pas introduit de biais par ces choix de traitement des individus et avons uniquement supprimé des erreurs.

# Regroupement des années de sortie

Il n'y a pas suffisamment de jeux avec comme Release Year < 2013, nous décidons donc de les regrouper en une seule classe. Nous regroupons aussi l'année 2026 avec l'année 2025 car il n'y a que 84 jeux parus en 2026 avant la création du dataset.

In [ ]:
years = pd.DatetimeIndex(pd.to_datetime(df["Release date"])).year
df["Year of Release"] = years
df["Year of Release"].value_counts()

In [ ]:
df.loc[df["Year of Release"] <= 2013, "Year of Release"] = 2013
df.loc[df["Year of Release"] == 2026, "Year of Release"] = 2025

df["Year of Release"] = df["Year of Release"].astype('str')
if "Release date" in df.columns :
    df = df.drop(columns="Release date")
df["Year of Release"].value_counts().sort_index()

Nous avons donc maintenant des classes qui contiennent toutes plus de 1500 jeux.

Nous remarquons, par la même occasion, que le nombre de jeux parus chaque année est croissant.

# Création des colonnes TagGenre

Ici on cherche à combiner les deux colonnes "Tags" et "Genres" car il y a beaucoup de duplicatas.

Les Tags et Genres étant stockés dans un string séparé par des virgules, nous les séparons en différentes colonnes, c'est une forme de one hot encoding.

In [ ]:
def merge_and_clean(row):
    tags = str(row['Tags']).split(',') if pd.notna(row['Tags']) else[]
    genres = str(row['Genres']).split(',') if pd.notna(row['Genres']) else[] 
    
    tous_les_mots =[mot.strip() for mot in (tags + genres) if mot.strip()]
    mots_uniques = set(tous_les_mots)
    return ','.join(mots_uniques)
    
df2 = df.copy(deep = True)
combined_series = df2.apply(merge_and_clean, axis=1)
combined_dummies = combined_series.str.get_dummies(sep=',').astype(bool)
combined_dummies.columns =[f"TagGenre_{c}" for c in combined_dummies.columns]
df2 = pd.concat([df2, combined_dummies], axis=1)
print(df2.head())

On sépare les colonnes Developers et Publishers car elles possèdent trop de valeurs uniques et donc provoqueraient une explosion du nombre de colonnes lors de leur one hot encoding pour les méthodes de MCA et MFA.

In [ ]:
Optionels = ["Developers", "Publishers"]
df2 = df2.drop(columns=Optionels)
df2.info(verbose=all)

In [ ]:
Categories_dummies = df2['Categories'].str.get_dummies(sep=',').astype(bool)
Categories_dummies.columns = [f"Categories_{c.strip()}" for c in Categories_dummies.columns] #one hot encoding de Categories
df2 = pd.concat([df2, Categories_dummies], axis=1)
print(df2.head())

In [ ]:
df2 = df2.drop(columns=["Tags", "Genres", "Categories"])

Nous avons donc séparé en multiples colonnes les variables "Tags", "Genres" et "Categories", en supprimant au maximum les duplicatas entre Tags et Genres.

## Outliers

Nous nous intéressons maintenant aux outliers et potentielles transformations nécessaires de colonnes.

Ici on considère comme Outliers les individus qui ne respectent pas la règle du 1.5 * IQR (1.5 * l'écart inter-quantile)

In [ ]:
NUM_COLS = df2.select_dtypes(include=["number", "int64"])
outlier_summary = plot_outliers(df2, NUM_COLS.columns)

On observe une grande proportion d'outliers sur certaines variables pour différentes raisons :

La majorité des variables ont en fait un trop grand nombre de 0, ce qui force la borne basse et haute de la détection d'outlier à la même valeur : 0.
Cela provoque la détection de toute valeur non nulle comme étant un outlier.

Il est possible de changer le critère d'outlier pour être par exemple : Le 1% des valeurs les plus extrêmes sont des outliers.

Nous pouvons aussi réaliser la même étude uniquement en considérant les valeurs non nulles pour le calcul des quantiles, ce que nous décidons de faire.

In [ ]:
NUM_COLS = df2.select_dtypes(include=["number", "int64"])
outlier_summary = plot_outliers_non_nuls(df2, NUM_COLS.columns)

On décide pour la suite que nous appliquerons des transformations log sur :

- Positive
- Negative
- Recommandation
- Les temps de jeux forever (car les valeurs maximales sont très grandes). 

D'autres transformations seront discutées lors de l'ACP (Peak CCU, Price, Language count et DLC count).

Nous remarquons aussi que certaines variables quantitatives sont presque toujours égale à 0 tel que les **playtime two weeks** et **Metacritic score**. La prédiction de la présence ou non d'un score Metacritic sera étudier dans le notebook R.

## Les plus grand Outliers par variable

Nous trouvons intéressant de regarder si les plus grands outils pour chaque variable sont les mêmes jeux afin de savoir si certains jeux ont des comportements très distincts des autres.

In [ ]:
cols_of_interest = [
    "Median playtime forever",
    "Average playtime forever", 
    "Recommendations",
    "Peak CCU",
    "Positive",
    "Negative",
    "DLC count"
]

def get_top_outliers(df, cols, top_n=5):
    for col in cols:
        serie = df[col]
        Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
        IQR = Q3 - Q1
        mask_outliers = (serie < Q1 - 1.5 * IQR) | (serie > Q3 + 1.5 * IQR)
        
        # Récupérer les outliers avec leur nom
        outliers = df[mask_outliers][["Name", col]].copy()
        outliers = outliers.sort_values(col, ascending=False).head(top_n)
        
        print(f"\n{'='*50}")
        print(f"Top {top_n} outliers pour {col}")
        print(f"{'='*50}")
        print(outliers.to_string(index=False))

get_top_outliers(df2, cols_of_interest, top_n=3)

Nous trouvons des outliers très similaires pour **Positive**, **Negative**, **Recommendations** et **Peak CCU**, les variables sont donc probablement très fortement liées, même chose pour les **Playtime Forever**. Nous remarquons aussi que les outliers de **DLC count** sont en partie les jeux d'une même franchise ("Fantasy Grounds" et "RPG maker").

## Les jeux les plus "détestés"

Nous regardons maintenant les jeux ayant la plus grande différence entre le nombre d'avis positif et le nombre d'avis négatif.

In [ ]:
cols_of_interest = [
    "Positive",
    "Negative",
]
df_temp = df2[["Name","Positive","Negative"]].copy()
df_temp["Negative - Positive"] = (df2["Negative"] - df2["Positive"])
outliers = df_temp.sort_values("Negative - Positive", ascending=False).head(5)
print(f"{'='*50}")
print(outliers.to_string(index=False))

Après quelques recherches, ces jeux sont pour la plupart cibles de "review bombing" pour une raison particulière. 

Par exemple pour "Kerbal Space Program 2" il s'agit de l'annonce de la fermeture du studio pour raison financière qui a provoqué ces avis. 

Pour "Mirror 2: Project X" il s'agit d'une protestation des joueurs sur un choix du studio de ne pas intégrer de contenu pour adultes dans le jeu alors que le prequel en contenait.

## Application des transformations log

Nous appliquons ici les transformations mentionnées dans la partie Outliers.

In [ ]:
df3 = df2.copy(deep = True)
df3["Positive"] = np.log2(df2["Positive"] + 1)
df3["Negative"] = np.log2(df2["Negative"] + 1)
df3["Recommendations"] = np.log2(df2["Recommendations"] + 1)
df3["Average playtime forever"] = np.log2(df2["Average playtime forever"] + 1)
df3["Median playtime forever"] = np.log2(df2["Median playtime forever"] + 1)

# Etude des corrélations entre variables Quantitatives

Nous cherchons maintenant à trouver (et confirmer) quelles sont les variables quantitatives corrélées ensembles.

Cette étude est faite sur deux datasets différents :
- Le premier possède uniquement les tranformations précédentes
- Le deuxième met en place des transformations additionnelles sur d'autres variables

L'objectif est de séparer les variables en 'groupes' corrélés ensemble et comprendre mieux leurs 'types' de corrélation.

## Dataset sans les autres log transform

In [ ]:
data_pca_2 = df3.select_dtypes(include=["number", "int64"]) #
scaler = StandardScaler()
scaled_df3 = scaler.fit_transform(data_pca_2)
pca_2 = PCA()
pca_df3 = pca_2.fit_transform(scaled_df3) 
explained_variance_2 = pca_2.explained_variance_ratio_

#feature_names = data_pca_2.columns.tolist()
#plot_correlation_circle(pca, feature_names, dim_pairs=[(0,1), (0,2), (1,2)])

corr_matrix = data_pca_2.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

## Dataset avec les autres log transform sur Peak CCU, DLC count, Price et Language Count

In [ ]:
df3_withOtherLogs = df3.copy(deep = True)
df3_withOtherLogs["Peak CCU"] = np.log2(df3["Peak CCU"] + 1)
df3_withOtherLogs["DLC count"] = np.log2(df3["DLC count"] + 1)
df3_withOtherLogs["Price"] = np.log2(df3["Price"] + 1)
df3_withOtherLogs["Language Count"] = np.log2(df3["Language Count"] + 1)

In [ ]:
data_pca = df3_withOtherLogs.select_dtypes(include=["number", "int64"])
scaler = StandardScaler()
scaled_df3 = scaler.fit_transform(data_pca)
pca = PCA()
pca_df3 = pca.fit_transform(scaled_df3) 
explained_variance = pca.explained_variance_ratio_

feature_names = data_pca.columns.tolist()

In [ ]:
corr_matrix = data_pca.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

Il est intéressant de noter que les corrélations sont très différentes entre les datasets avec/sans log supplémentaires sur **Price**, **Language Count**, **Peak CCU** et **DLC count**.

Ça laisse donc sous-entendre que les liens entre les variables sont probablement assez complexes et pas simplement linéaires ou bien que des valeurs extrêmes bruitent l'ACP sans log.

**La Popularitée**

Le pic de joueurs simultanés est fortement corrélé aux avis positifs/négatifs, aux recommendations, au playtime forever, et dans une moindre mesure au Metacritic score (0.36) et au nombre de DLC (0.37). Un jeu populaire génère donc globalement plus d'engagement sous toutes ses formes.

**Recommendations et temps de jeu**

Le groupe **Recommendations (avec positive et negative), et les playtime forever** forme un cluster très cohérent (corrélations entre 0.7 et 0.99). Les joueurs qui recommandent un jeu y passent aussi beaucoup de temps ce qui est logique : on recommande ce à quoi on a vraiment joué.

**Avis positifs et négatifs symétriques**

**Positive** et **Negative** sont corrélés à extrèmement fortement. Un jeu très joué accumule mécaniquement beaucoup d'avis dans les deux sens. Ce sont davantage des indicateurs de volume d'avis car les avis ont tendance à converger vers moyen (autant de positifs que de négatifs).

**Playtime Two weeks**

Les variables **Average** et **Median playtime two weeks** sont très fortement corrélées entre elles (0.97), tout comme leurs équivalents **forever** (0.99). Mais le playtime **two weeks** et le playtime **forever** sont faiblement liés, ce qui suggère que le temps de jeu récent ne reflète pas nécessairement le temps de jeu cumulé sur le long terme. Un jeu ancien très joué peut avoir peu d'activité récente, et inversement.

**Prix et Language Count peu corrélés**

Le prix et le nombre de langages, bien que corrélés à Recommendations, Peak CCU et les Playtime Forever, les corrélations sont très faibles < 0.25.

**Achivements isolé**

Toutes les corrélations de **Achivements** sont proches de 0, on considère donc qu'il s'agit d'une information complètement différente des autres variables.

**Conclusion générale**

Les variables se regroupent en plusieurs blocs :
- **Popularité** : Peak CCU, Positive, Negative, Recommendations, Average/Median playtime foreverUser score
- **Engagement court terme** : Average/Median playtime two weeks, (corrélations > 0.97 entre eux)
- **Prix** : Price et Language Count sont faiblement corrélées entre elles et aussi avec le bloc de 'Popularité'. 
- **Variables indépendantes** : Achievements est presque complètement independant des autres variables.

Le Metacritic score et le nombre de DLC semblent jouer un rôle modéré sur la popularité.

## Représentation des données dans la PCA

Nous souhaitons maintenant savoir ce que représentent les axes dans la PCA ainsi que le nombre de dimensions concernées.

In [ ]:
plt.plot(pca.explained_variance_ratio_.cumsum() * 100)
plt.grid()
plt.title("Pourcentage de variance expliquée selon le nombre de dimensions concernées")
plt.show()

On observe une réduction de dimension importante : pour conserver 80 % de la variance, il suffit de conserver 6 dimensions au lieu de 14, il s'agit d'une compression d'environ 2.3x de l'information de variables quantitatives.

In [ ]:
plot_correlation_circle(pca, feature_names, dim_pairs=[(0,1), (0,2), (1,2)])

La PC1 est assez compliquée à lire mais semble encoder l'engagement/popularité d'un jeu, avec notamment le **Peak CCU**, les **Recommendations** (Avex les avis positifs et négatifs), les **Playtime forever**. 

La PC2 encode clairement les variables **Playtime two weeks**.

La PC3 semble, elle, indiquer le prix du jeu avec **Price** et **Discount** (Discount n'est pas réellement encodé dans la PC3 mais la contre-corrélation avec **Price** la fait ressortir du lot). Cependant la PC3 n'explique déjà plus que seulement 7 % de la variance des données. Il semble que **Achivements** soit aussi légèrement encodé dans la PC3 mais très peu (comme **Discount**)

### Individuals Factor Map 

In [ ]:
## selecting the first 9 principal components

i = 9
pca_games = pca_df3[:,0:i-1]

expl_var = np.sum(pca.explained_variance_ratio_[:i-1])
print(f"The first {i} PC represent {expl_var*100:.1f}% of the variance")

In [ ]:
import matplotlib

fig, axes = plt.subplots(2,3,figsize = (12,8))

pairs = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]

for idx, (i,j) in enumerate(pairs):

        ax = axes.flatten()[idx]
        ax.scatter(pca_games[:,i],  pca_games[:,j], s = 1, alpha = 0.4, color = "steelblue")

        ax.set_title("Individuals factor map — PCA")
        ax.set_xlabel(f"PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)")
        ax.set_ylabel(f"PC{j+1} ({pca.explained_variance_ratio_[j]*100:.1f}%)")
        ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
        ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

# Clustering on Individuals

In this section, we will perform K-means clustering on the numerical data, in order to find structures in the steam market of video games.

In [ ]:
df_all = df2.copy(deep = True)

### K-means on numerical data

In [ ]:
np.random.seed(42)

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path

### k means over pca data for 2 to 30 number of clusters 

cache_path = Path("UMAPs_Silhouette_scores/sil_scores.npy")
cache_path.parent.mkdir(exist_ok=True)

if cache_path.exists():
    
    sil_scores = np.load(cache_path)

else:

    sil_scores = [0,0]

    for k in range(2, 30):
        
        kmeans = KMeans(n_clusters=k, init='k-means++', n_init='auto', max_iter=100, random_state=42)
        
        labels = kmeans.fit_predict(pca_games)
        
        sil_score = silhouette_score(pca_games, labels, sample_size= 24000, random_state=42)  #### calculating on a subsample to reduce computational costs

        print(sil_score)

        sil_scores.append(sil_score)
    
    np.save(cache_path, sil_scores)

        

In [ ]:
x = np.arange(30)

fig, ax = plt.subplots(figsize = (7,5))

ax.plot(x, sil_scores, color="Steelblue", linewidth=2)

ax.set_xlabel("Nombre de clusters dans K-Means")
ax.set_ylabel("Score de silhouette")
ax.set_title("Score de silhouette pour K-Means appliqué aux données PCA")
ax.set_xlim(0,30)

plt.tight_layout()
plt.show()


So we will be looking at Kmeans clustering results for K = 2 and k = 11.

In [ ]:
np.random.seed(42)

cl_sizes = [2,11]

cluster_partitions = []

i = 0

for K in cl_sizes: 

    kmeans_games = KMeans(n_clusters=K, init="k-means++", n_init="auto", random_state = 42)
    cluster_partitions.append(kmeans_games.fit_predict(pca_games))

In [ ]:
### Plotting the Partitions for two different Ks on the individuals faactor map


fig, axes = plt.subplots(4,3, figsize=(16,20))

pairs = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]

for index, partition in enumerate(cluster_partitions):

    K = cl_sizes[index]

    cmap = plt.get_cmap('tab20', K)

    for idx, (i,j) in enumerate(pairs):

        ax = axes.flatten()[6 * index + idx]

        sc = ax.scatter(pca_games[:,i],pca_games[:,j],c=partition,s=1,alpha=0.6,cmap=cmap
        )

        ax.set_title(f"K-Means (K={K}) sur CP{i+1} et CP{j+1}")

        ax.set_xlabel(f"CP{i+1}")
        ax.set_ylabel(f"CP{j+1}")

        ax.axhline(0, color="gray", linewidth=0.8,linestyle="--", alpha=0.5)

        ax.axvline(0, color="gray", linewidth=0.8,linestyle="--", alpha=0.5)

        ax.spines[["top", "right"]].set_visible(False)

        ax.grid(True, linestyle="--", alpha=0.3)

        fig.colorbar(sc,ax=ax,ticks=range(K),label="Cluster de jeux")

plt.tight_layout()
plt.show()

Very difficult to interpret! Only for K=2 K-means apperently yields two clusters where one contains games close to the origin. A umap projection might yield better results.

In [ ]:
np.random.seed(42)

### Calculating the UMAP Projection of PCA Data
import umap.umap_ as umap 

cache_path = Path("UMAPs_Silhouette_scores/umap_pca_games.npy")
cache_path.parent.mkdir(exist_ok=True)

if cache_path.exists():
        games_pca_umap = np.load(cache_path)

else:
    reducer = umap.UMAP(n_neighbors= 15, min_dist=0.1, random_state=42)

    games_pca_umap = reducer.fit_transform(pca_games)

    np.save(cache_path, games_pca_umap)

In [ ]:
## Plot the Clusters for both partitions on the UMAP Projection 

fig, axes = plt.subplots(1,2, figsize=(20,9))

for index, partition in enumerate(cluster_partitions):
    
    K = cl_sizes[index]

    cmap = plt.get_cmap('tab20',K)

    ax = axes.flatten()[index]
    sc = ax.scatter(games_pca_umap[:,0], games_pca_umap[:,1], c = partition, s = 4, alpha = 0.6, cmap = cmap)

    ax.set_title(f"Projection UMAP des données PCA colorée par le clustering K-Means avec K = {K}")
    ax.set_xlabel("Dimension UMAP 1")
    ax.set_ylabel("Dimension UMAP 2")

    fig.colorbar(sc, ax=ax, ticks=range(K), label= "Cluster de jeux")

plt.tight_layout()
plt.show()

These clusters are no longer convex, even though K-Means creates convex clusters, so this low-dimensional (2D) umap representation is only an approach to reality!

In [ ]:
# Number of Games in each cluster depending on 2 different K's

fig, axes = plt.subplots(1,2,figsize = (12,5))

for i, partition in enumerate(cluster_partitions):

    cmap = plt.get_cmap('tab20', cl_sizes[i])
    
    ax = axes.flatten()[i]
    ax.bar(*np.unique(partition, return_counts=True), color=cmap.colors)
    ax.set_ylabel("Nombre de jeux par cluster")
    ax.set_xlabel("Cluster")
    ax.set_title(f"Nombre de clusters : {cl_sizes[i]}")

plt.tight_layout()
plt.show()



Really unevenly distributed clusters, but maybe the small ones are really niche in some variable? We will now further investigate on this!

### Creation of the column "owners_num" to sort the "Estimated Owners" values as it is of dtype object

In [ ]:
def parse_owners(x): 
    low, high = x.replace(",", "").split(" - ")
    return (int(low)+ int(high)) / 2

df_all["owners_num"] = df_all["Estimated owners"].apply(parse_owners)

We now want to do a cluster analysis and look at the properties of each cluster for the two partitions.

We try to answer the following questions: 

    1. What patterns/structures/similarities between games are there within each cluster for the numerical variables?

    2. Does the numerical data contain information also about the qualitative data, even though it was not part of the data on which the clustering was performed?

In [ ]:
## for explanation of these functions, take a look into the folder "Fonctions"

for partition in cluster_partitions:

    frame = partition_analysis(df_all, partition)

    results = collect_cluster_results(frame)

    labels = make_labels(results)

    variabs_to_investigate = list(labels.keys())

    for var in variabs_to_investigate:

        if var in ["categories", "genres", "top_games", "required_age", "year", "estimated_owners"]:
            plot_one_Variable_on_umap(var, labels.get(var),games_pca_umap, "PCA des données numériques", partition, False)
        else:
            plot_one_Variable_on_umap(var,labels.get(var),games_pca_umap, "PCA des données numériques", partition, True)

### Summary on the Partition Investigation:

#### For k = 2

The cluster partition obtained separates relatively well games that are barely played or non-successful from those with a small to large player base. Indicators supporting this observation are especially the variables median_playtime_forever, peak_ccu, recommendations, and release_year. 

The first three variables show a clear correlation with the success of a game, whereas for release_year the relationship is less intuitive. While 2016 and 2017 were among the top three most frequent release years in the dark blue cluster, 2025, 2024, and 2023 were the most frequent years in the light blue cluster. This could indicate either that games need time to be discovered by players or that in recent years a large number of poorly designed games have been released that are now barely played.

In summary, this partition remains rather broad and vague, but it nevertheless outlines relatively well the properties of successful games.

#### For k = 11

Now the clusters become more precise and distinct. Still, this is a partition of the same data, which is why the general structure is maintained: poorly performing games in the lower part of the projection (colors: pink, green, red, dark blue, and grey), which mostly made up the dark blue cluster in the partition above, and well-performing games in the upper right center (colors: orange, brown, and light blue), which previously formed the light blue cluster, remain clearly separated. However, within these broad groups, more detailed distinctions now emerge.

In the green and pink clusters, one mostly finds recently published free-to-play games that are almost unknown and have very low player engagement. In the dark blue cluster, games with sexual content appear to be gathered. Within the red cluster, which shows an unusually high language_count for such unsuccessful games, there are mostly puzzle and hidden-object games involving various animals. These games are typically recently published, quickly played through, and at least appear to be available in a large number of languages.

The better-performing clusters consist of rather brutal first-person shooter classics (light blue), highly popular multiplayer and competitive online games such as Counter-Strike 2, Dota 2, Apex Legends, or PUBG (brown and orange), as well as clusters containing more established sandbox, economy, and co-op games. These clusters generally show substantially higher values for variables such as median_playtime_forever, peak_ccu, recommendations, and language_count, indicating both larger player bases and stronger long-term player retention.

Overall, this partition appears considerably more informative than the previous one. While the earlier partition mainly separated successful from unsuccessful games, the higher number of clusters now additionally reveals more specific subgroups and niches within the Steam ecosystem. Especially interesting is the observation that games with very similar numerical properties often also share thematic or gameplay-related similarities, even though the clustering itself was performed only on quantitative variables.

## Now the inverse, do qualitative variables say something about the Quantitatives?

In [ ]:
df_spectral = df_all.copy(deep=True)

cat_cols = df_spectral.select_dtypes(exclude="number").columns

df_spectral = df_spectral[cat_cols].copy()

df_spectral.columns

In [ ]:
df_spectral = df_spectral.drop(columns=['Estimated owners', 'Name', 'Supported languages', 'Full audio languages', 'Categories', 'Genres', 'Tags'])

df_spectral.columns

In [ ]:
df_cat_new = pd.get_dummies(df_spectral[['Year of Release']], drop_first=False)

df_spectral = df_spectral.drop(columns=['Year of Release'])

df_mca_spectral = pd.concat(
    [
        df_spectral,
        df_cat_new
    ],
    axis=1
)

df_mca_spectral.columns  #### we only keep the qualitative variables

In [ ]:
import prince

mca = prince.MCA(
    n_components=100,
    n_iter=20,
    copy=True,
    check_input=True,
    engine="sklearn",
    random_state=42
)

mca = mca.fit(df_mca_spectral)

mca.eigenvalues_summary

In [ ]:
inertia = mca.percentage_of_variance_

# Scree Plot
plt.figure(figsize=(12,7))

plt.plot(
    range(1, len(inertia)+1),
    inertia,
    marker='o'
)

plt.xlabel("Composante MCA")
plt.ylabel("Inertie expliquée (%)")
plt.title("Scree plot de la MCA")

plt.xticks(range(1, len(inertia)+1, 5))

plt.grid(True)
plt.show()

In [ ]:
X_mca = mca.row_coordinates(df_mca_spectral).iloc[:, :20].values

The 1/p rule (keeping all components with more than 1/p inertia) with p=500 does not really make sense here, so we take the first 20, as from there on the curve only slowly decreases.

In [ ]:
### umap projection of MCA data

cache_path = Path("UMAPs_Silhouette_scores/mca_umap.npy")
cache_path.parent.mkdir(exist_ok=True)

if cache_path.exists():
        
    mca_umap = np.load(cache_path)

else:
    reducer = umap.UMAP(n_neighbors=30,min_dist=0.1,n_components=2,random_state=42)

    mca_umap = reducer.fit_transform(X_mca)

    np.save(cache_path,mca_umap)


The Clusters based on Spectral Clustering in Graphs did not yield a interpretable outcome!

Maybe we can find similaritys of quantitavie variables through other methods!

In [ ]:
### umap projection of MCA data 

df_log_scaled = df3_withOtherLogs.copy(deep=True)

df_quant_log = df_log_scaled.select_dtypes(include=["number", "int64"]).copy()

df_quant_log.columns

df_quant_log.dtypes

In [ ]:
### we want to find peaks/structures of the quantitative variables in the UMAP projection

### we normalize the data to have min 0 and max 1 for coloring

def transform_0_1(df_quant):

    for var in df_quant.columns: 

        df_temp = df_quant[var]

        df_quant[var] = (df_temp - np.min(df_temp)) / (np.max(df_temp) - np.min(df_temp))

    return df_quant

In [ ]:
df_quant_log_scaled = transform_0_1(df_quant_log)

In [ ]:
for var in df_quant_log_scaled.columns:

    print("")
    print(var)
    df_temp = df_quant_log_scaled[var]
    print(df_temp.min(), df_temp.mean(), df_temp.median() ,df_temp.max())

In [ ]:
def plot_var_on_umap(mca_umap, var, df_quant_log_scaled, color):

    fig, ax = plt.subplots(figsize=(14,12))
  
    sc = ax.scatter(mca_umap[:,0], mca_umap[:,1], c = df_quant_log_scaled[var], cmap = color, s = 8, alpha = 0.6)

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(var)

    ax.set_title(f"Projection UMAP des 20 premières composantes MCA colorée par la variable transformée entre 0 et 1 : {var}")
    ax.set_xlabel("Dimension UMAP 1")
    ax.set_ylabel("Dimension UMAP 2")

    plt.tight_layout()
    plt.show()

In [ ]:
for var in df_quant_log_scaled.columns:

    plot_var_on_umap(mca_umap, var, df_quant_log_scaled, "viridis")

In [ ]:
def plot_qual_on_umap(mca_umap, var, data):

    fig, ax = plt.subplots(figsize=(14,12))

    colors = pd.to_numeric(data[var], errors="coerce")

    sc = ax.scatter(
        mca_umap[:,0],
        mca_umap[:,1],
        c=colors,
        cmap="plasma",
        s=8,
        alpha=0.7
    )

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(var)

    ax.set_title(f"UMAP Projection colored by {var}")
    ax.set_xlabel("UMAP Dim1")
    ax.set_ylabel("UMAP Dim2")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_qual_on_umap(mca_umap, "Year of Release", df_all)

This is not really clear here. For the quantitative variables plotted on the UMAP projection of the MCA of the qualitative variables, almost no clear structure was observable. Well-performing games appear to be distributed almost randomly within the large central “cluster” or structure. This could indicate that the chosen methods were not fully appropriate for capturing such relationships, that there is no simple correlation structure detectable through MCA, or that relevant information was lost during the UMAP projection.

Nevertheless, some smaller localized structures and thematic groupings can still be observed, suggesting that at least parts of the qualitative information are represented in the embedding. Overall, however, the separation remains substantially weaker than for the clustering performed directly on the quantitative variables.